In [1]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
movies = pd.read_csv('movies.csv')

print(train.head())
print(train.shape)

print(test.head())
print(test.shape)

print(movies.head())
print(movies.shape)

   userId  movieId  rating   timestamp
0    5163    57669     4.0  1518349992
1  106343        5     4.5  1206238739
2  146790     5459     5.0  1076215539
3  106362    32296     2.0  1423042565
4    9041      366     3.0   833375837
(10000038, 4)
   userId  movieId
0       1     2011
1       1     4144
2       1     5767
3       1     6711
4       1     7318
(5000019, 2)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
(62423, 3)


In [2]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000038 entries, 0 to 10000037
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 305.2 MB


In [3]:
train.describe()

,userId,movieId,rating,timestamp
count,1.000004e+07,1.000004e+07,1.000004e+07,1.000004e+07
mean,8.119909e+04,2.138911e+04,3.533395e+00,1.215677e+09
std,4.679359e+04,3.919578e+04,1.061124e+00,2.268921e+08
min,1.000000e+00,1.000000e+00,5.000000e-01,7.896520e+08
25%,4.051000e+04,1.197000e+03,3.000000e+00,1.011742e+09
50%,8.091400e+04,2.947000e+03,3.500000e+00,1.199019e+09
75%,1.215790e+05,8.630000e+03,4.000000e+00,1.447242e+09
max,1.625410e+05,2.091710e+05,5.000000e+00,1.574328e+09


In [4]:
train.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [5]:
print("Number of users:", train['userId'].nunique())
print("Number of movies:", train['movieId'].nunique())

print("Minimum rating:", train['rating'].min())
print("Maximum rating:", train['rating'].max())

Number of users: 162541
Number of movies: 48213
Minimum rating: 0.5
Maximum rating: 5.0


In [6]:
train['rating'].value_counts().sort_index()

rating
0.5     157571
1.0     311213
1.5     159731
2.0     656821
2.5     505578
3.0    1959759
3.5    1270642
4.0    2652977
4.5     880516
5.0    1445230
Name: count, dtype: int64

In [7]:
train.duplicated().sum()

np.int64(0)

In [8]:
print(train.shape)
print(test.shape)

print("Users:", train['userId'].nunique())
print("Movies:", train['movieId'].nunique())

print("Ratings:", train['rating'].count())

(10000038, 4)
(5000019, 2)
Users: 162541
Movies: 48213
Ratings: 10000038


In [9]:
train.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [10]:
train['rating'].value_counts().sort_index()

rating
0.5     157571
1.0     311213
1.5     159731
2.0     656821
2.5     505578
3.0    1959759
3.5    1270642
4.0    2652977
4.5     880516
5.0    1445230
Name: count, dtype: int64

In [11]:
print(train.shape)
print(test.shape)

print("Users:", train['userId'].nunique())
print("Movies:", train['movieId'].nunique())

(10000038, 4)
(5000019, 2)
Users: 162541
Movies: 48213


In [12]:
train.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [13]:
%pip install scikit-surprise

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from surprise import Dataset, Reader, SVD

print("Surprise installed successfully!")

Surprise installed successfully!


In [15]:
train['userId'].value_counts().head(10)

userId
72315     12952
80974      3680
137293     3588
33844      3208
20055      3050
109731     2671
49403      2670
92046      2579
110971     2288
30879      2284
Name: count, dtype: int64

In [16]:
train['movieId'].value_counts().head(10)

movieId
318     32831
356     32383
296     31697
593     29444
2571    29014
260     27560
480     25518
527     24004
110     23722
2959    23536
Name: count, dtype: int64

In [17]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

reader = Reader(rating_scale=(0.5, 5))

data = Dataset.load_from_df(
    train[['userId', 'movieId', 'rating']],
    reader
)

trainset, valset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

model = SVD(random_state=42)

model.fit(trainset)

predictions = model.test(valset)

rmse(predictions)

RMSE: 0.8335


np.float64(0.8334747134562795)

In [18]:
from surprise import SVD
from surprise.accuracy import rmse

model = SVD(
    n_factors=150,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model.fit(trainset)

predictions = model.test(valset)

rmse(predictions)

RMSE: 0.8387


np.float64(0.8387386736613439)

In [19]:
reader = Reader(rating_scale=(0.5, 5))

data = Dataset.load_from_df(
    train[['userId', 'movieId', 'rating']],
    reader
)

full_trainset = data.build_full_trainset()

final_model = SVD(
    n_factors=150,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

final_model.fit(full_trainset)

In [20]:
predictions = []

for _, row in test.iterrows():

    pred = final_model.predict(
        row['userId'],
        row['movieId']
    )

    predictions.append(pred.est)

In [21]:
submission = test.copy()

submission['Id'] = (
    submission['userId'].astype(str)
    + '_'
    + submission['movieId'].astype(str)
)

submission['rating'] = predictions

submission = submission[['Id', 'rating']]

submission.to_csv(
    'submission.csv',
    index=False
)

submission.head()

,Id,rating
0,1_2011,3.600872
1,1_4144,4.346031
2,1_5767,3.664234
3,1_6711,3.461639
4,1_7318,3.377404


In [22]:
submission = test.copy()

submission['Id'] = (
    submission['userId'].astype(str)
    + '_'
    + submission['movieId'].astype(str)
)

submission['rating'] = predictions

submission = submission[['Id', 'rating']]

submission.to_csv('submission.csv', index=False)

print(submission.head())
print(submission.shape)

       Id    rating
0  1_2011  3.600872
1  1_4144  4.346031
2  1_5767  3.664234
3  1_6711  3.461639
4  1_7318  3.377404
(5000019, 2)


In [23]:
import psutil

print(psutil.virtual_memory().total / (1024**3))

15.88339614868164


In [24]:
import pandas as pd

batch_size = 500000

submission_parts = []

for start in range(0, len(test), batch_size):

    end = min(start + batch_size, len(test))

    batch = test.iloc[start:end].copy()

    batch['rating'] = [
        model.predict(uid, iid).est
        for uid, iid in zip(
            batch['userId'],
            batch['movieId']
        )
    ]

    batch['Id'] = (
        batch['userId'].astype(str)
        + '_'
        + batch['movieId'].astype(str)
    )

    submission_parts.append(
        batch[['Id', 'rating']]
    )

    print(f"Processed {end:,} rows")

Processed 500,000 rows
Processed 1,000,000 rows
Processed 1,500,000 rows
Processed 2,000,000 rows
Processed 2,500,000 rows
Processed 3,000,000 rows
Processed 3,500,000 rows
Processed 4,000,000 rows
Processed 4,500,000 rows
Processed 5,000,000 rows
Processed 5,000,019 rows


In [25]:
submission = pd.concat(
    submission_parts,
    ignore_index=True
)

submission.to_csv(
    'submission.csv',
    index=False
)

print(submission.shape)
submission.head()

(5000019, 2)


,Id,rating
0,1_2011,3.281626
1,1_4144,4.224073
2,1_5767,3.400269
3,1_6711,3.976283
4,1_7318,3.275616


In [26]:
submission.isnull().sum()

Id        0
rating    0
dtype: int64